In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


In [ ]:
QUERY_DIM=384
HIDDEN_DIM_1 = 512
HIDDEN_DIM_2 = 256
OUTPUT_DIM = 128

In [ ]:
class PHDTower(nn.Module):
  def __init__(self, vocab_dict, dropout=0.2):
    super().__init__()

    self.dense_layers = nn.Sequential(
        nn.Linear(QUERY_DIM, HIDDEN_DIM_1),
        nn.LayerNorm(HIDDEN_DIM_1),
        nn.ReLU(),
        nn.Dropout(dropout),

        nn.Linear(HIDDEN_DIM_1, HIDDEN_DIM_2),
        nn.LayerNorm(HIDDEN_DIM_2),
        nn.ReLU(),

        nn.Linear(HIDDEN_DIM_2, OUTPUT_DIM)
    )

    def forward(self, data):
      all_embs = ""
      out = self.dense_layers(all_embs)
      return F.normalize(out, p=2, dim=-1) #(batch, 128)


In [ ]:
CANDIDATE_DIM = 384
HIDDEN_DIM_1 = 512
HIDDEN_DIM_2 = 256
OUTPUT_DIM = 128

In [ ]:
class ProfTower(nn.Module):
  def __init__(self, vocab_dict, dropout=0.2):
    super().__init__()

    self.dense_layers = nn.Sequential(
        nn.Linear(CANDIDATE_DIM, HIDDEN_DIM_1),
        nn.LayerNorm(HIDDEN_DIM_1),
        nn.ReLU(),

        nn.Linear(HIDDEN_DIM_1, HIDDEN_DIM_2),
        nn.LayerNorm(HIDDEN_DIM_2),
        nn.ReLU(),

        nn.Linear(HIDDEN_DIM_2, OUTPUT_DIM)
    )

  def forward(self, data):
    all_embs = ""
    out = self.dense_layers(all_embs)
    return F.normalize(out, p=2, dim=-1)




In [ ]:
class TwoTower(nn.Module):
  def __init__(self, vocab_dict, dropout=0.2):
    super().__init__()

    self.query_tower = PHDTower(vocab_dict, dropout)
    self.candidate_tower = ProfTower(vocab_dict, dropout)

    self.apply(self.init_weights_xavier)

  def init_weights_xavier(self, m):
    if isinstance(m, nn.Linear):
      nn.init.xavier_uniform_(m.weight)
      if m.bias is not None:
        nn.init.zeros_(m.bias)

  def forward(self, data):
    query_embs = self.query_tower(data)
    candidate_embs = self.candidate_tower(data)
    return query_embs, candidate_embs




In [ ]:
# pymupdf extractor
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 70.7 MB/s eta 0:00:00


In [ ]:
import pymupdf  # PyMuPDF

def extract_text(pdf_path):
    doc = pymupdf.open(pdf_path)
    full_text = []

    for page_num in range(len(doc)):
        page = doc[page_num]
        full_text.append(page.get_text())

    return "\n".join(full_text)

extracted_content = extract_text("/content/BTP_Implementation_plan.pdf")
# print(extracted_content)

In [ ]:
from IPython.display import display, Markdown
# display(Markdown(extracted_content))

In [ ]:
!pip install pymupdf4llm

In [ ]:
import pymupdf4llm
from pathlib import Path

# 1. Convert the PDF to a Markdown string
md_text = pymupdf4llm.to_markdown("/content/BTP_Implementation_plan.pdf")

# 2. Save it to a file (optional)
# Path("output.md").write_bytes(md_text.encode())

# print(md_text)


=== Document parser messages ===
Using Tesseract for OCR processing.
